# Notebook 04 — Complete Model Optimization & Balanced Selection

1. Baseline model training
2. SMOTE experiment
3. Fast hyperparameter tuning
4. Class-wise Precision / Recall / F1
5. Macro F1
6. Classification-threshold testing
7. Selection of the most balanced model




## 1. Imports and Configuration

In [1]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: d:\job acceptance prediction system


## 2. Load Notebook 03 Artifacts

Notebook 03 already performs:

- feature engineering
- train/test split
- preprocessing
- encoding



In [2]:
train_test_path = os.path.join(MODELS_DIR, "train_test_data.joblib")

if not os.path.exists(train_test_path):
    raise FileNotFoundError(
        f"{train_test_path} was not found. Run Notebook 03 first."
    )

data = joblib.load(train_test_path)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = np.asarray(data["y_train"])
y_test = np.asarray(data["y_test"])

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting distribution:")
print(pd.Series(y_test).value_counts().sort_index())


X_train: (39888, 102)
X_test : (9972, 102)
y_train: (39888,)
y_test : (9972,)

Training distribution:
0    14420
1    25468
Name: count, dtype: int64

Testing distribution:
0    3605
1    6367
Name: count, dtype: int64


# Part A — Baseline Models



In [3]:
baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_leaf=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),

    "SVM": SVC(
        probability=True,
        random_state=RANDOM_STATE
    )
}

print("\n".join(f"- {name}" for name in baseline_models))


- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- SVM


In [4]:
def evaluate_model(model, X_eval, y_eval, threshold=0.50):
    """Return overall and class-wise metrics for a binary classifier."""

    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_eval)[:, 1]
    else:
        scores = model.decision_function(X_eval)

    y_pred = (scores >= threshold).astype(int)

    report = classification_report(
        y_eval,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    return {
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1_Score": f1_score(y_eval, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_eval, scores),

        "Rejected_Precision": report["0"]["precision"],
        "Rejected_Recall": report["0"]["recall"],
        "Rejected_F1": report["0"]["f1-score"],

        "Accepted_Precision": report["1"]["precision"],
        "Accepted_Recall": report["1"]["recall"],
        "Accepted_F1": report["1"]["f1-score"],

        "Macro_F1": report["macro avg"]["f1-score"],
        "Weighted_F1": report["weighted avg"]["f1-score"],

        "Threshold": threshold
    }


In [5]:
trained_baseline = {}
baseline_rows = []

for name, model in baseline_models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_baseline[name] = model

    metrics = evaluate_model(
        model,
        X_test,
        y_test,
        threshold=0.50
    )

    metrics["Model"] = name
    metrics["Approach"] = "Baseline"
    baseline_rows.append(metrics)

baseline_df = pd.DataFrame(baseline_rows)

print("\nBaseline results:")
display(
    baseline_df[
        [
            "Model",
            "Rejected_Precision",
            "Rejected_Recall",
            "Rejected_F1",
            "Accepted_Precision",
            "Accepted_Recall",
            "Accepted_F1",
            "Macro_F1",
            "Accuracy",
            "ROC_AUC"
        ]
    ].sort_values("Macro_F1", ascending=False).round(4)
)


Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training SVM...

Baseline results:


,Model,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Accuracy,ROC_AUC
0,Logistic Regression,0.6014,0.3257,0.4225,0.6969,0.8778,0.7770,0.5997,0.6782,0.6894
3,Gradient Boosting,0.5885,0.2896,0.3882,0.6876,0.8853,0.7740,0.5811,0.6700,0.6859
2,Random Forest,0.5962,0.2777,0.3789,0.6860,0.8935,0.7761,0.5775,0.6709,0.6798
4,SVM,0.6110,0.2596,0.3644,0.6838,0.9064,0.7795,0.5720,0.6726,0.6634
1,Decision Tree,0.4982,0.3101,0.3823,0.6782,0.8232,0.7437,0.5630,0.6377,0.6190


# Part B — SMOTE Experiment


In [6]:
smote = SMOTE(
    random_state=RANDOM_STATE
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Original training shape:", X_train.shape)
print("SMOTE training shape   :", X_train_smote.shape)

print("\nOriginal distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nSMOTE distribution:")
print(pd.Series(y_train_smote).value_counts().sort_index())


Original training shape: (39888, 102)
SMOTE training shape   : (50936, 102)

Original distribution:
0    14420
1    25468
Name: count, dtype: int64

SMOTE distribution:
0    25468
1    25468
Name: count, dtype: int64


In [7]:
smote_models = {
    "SMOTE Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ),

    "SMOTE Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        max_depth=10,
        min_samples_leaf=5
    ),

    "SMOTE Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2
    ),

    "SMOTE Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ),

    "SMOTE SVM": SVC(
        probability=True,
        random_state=RANDOM_STATE
    )
}

trained_smote = {}
smote_rows = []

for name, model in smote_models.items():
    print(f"Training {name}...")
    model.fit(X_train_smote, y_train_smote)
    trained_smote[name] = model

    metrics = evaluate_model(
        model,
        X_test,
        y_test,
        threshold=0.50
    )

    metrics["Model"] = name
    metrics["Approach"] = "SMOTE"
    smote_rows.append(metrics)

smote_df = pd.DataFrame(smote_rows)

print("\nSMOTE results:")
display(
    smote_df[
        [
            "Model",
            "Rejected_Precision",
            "Rejected_Recall",
            "Rejected_F1",
            "Accepted_Precision",
            "Accepted_Recall",
            "Accepted_F1",
            "Macro_F1",
            "Accuracy",
            "ROC_AUC"
        ]
    ].sort_values("Macro_F1", ascending=False).round(4)
)


Training SMOTE Logistic Regression...
Training SMOTE Decision Tree...
Training SMOTE Random Forest...
Training SMOTE Gradient Boosting...
Training SMOTE SVM...

SMOTE results:


,Model,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Accuracy,ROC_AUC
0,SMOTE Logistic Regression,0.4981,0.6377,0.5593,0.7562,0.6361,0.6909,0.6251,0.6367,0.6878
3,SMOTE Gradient Boosting,0.5706,0.3911,0.4641,0.7074,0.8334,0.7652,0.6147,0.6735,0.6820
4,SMOTE SVM,0.5190,0.4660,0.4911,0.7142,0.7555,0.7342,0.6127,0.6508,0.6563
2,SMOTE Random Forest,0.5702,0.3603,0.4416,0.7003,0.8462,0.7664,0.6040,0.6706,0.6799
1,SMOTE Decision Tree,0.4960,0.4699,0.4826,0.7086,0.7297,0.7190,0.6008,0.6358,0.6391


# Part C — Hyperparameter Tuning



In [8]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

tuning_specs = {
    "Tuned Logistic Regression": (
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ),
        {
            "C": [0.01, 0.1, 1, 10],
            "solver": ["lbfgs"],
            "class_weight": [None, "balanced"]
        }
    ),

    "Tuned Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        {
            "n_estimators": [100, 150],
            "max_depth": [None, 8, 10],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"]
        }
    ),

    "Tuned Gradient Boosting": (
        GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        {
            "n_estimators": [100, 150],
            "learning_rate": [0.03, 0.05, 0.1],
            "max_depth": [2, 3],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    )
}

tuned_models = {}
tuning_rows = []

for name, (estimator, params) in tuning_specs.items():

    print(f"\nTuning {name}...")

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=params,
        n_iter=6,
        scoring="f1",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    tuned_models[name] = search.best_estimator_

    print("Best CV F1:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

    metrics = evaluate_model(
        search.best_estimator_,
        X_test,
        y_test,
        threshold=0.50
    )

    metrics["Model"] = name
    metrics["Approach"] = "Hyperparameter Tuned"
    metrics["Best_CV_F1"] = search.best_score_
    tuning_rows.append(metrics)

tuning_df = pd.DataFrame(tuning_rows)

print("\nTuned model results:")
display(
    tuning_df[
        [
            "Model",
            "Best_CV_F1",
            "Rejected_Precision",
            "Rejected_Recall",
            "Rejected_F1",
            "Accepted_Precision",
            "Accepted_Recall",
            "Accepted_F1",
            "Macro_F1",
            "Accuracy",
            "ROC_AUC"
        ]
    ].sort_values("Macro_F1", ascending=False).round(4)
)



Tuning Tuned Logistic Regression...
Best CV F1: 0.7785
Best parameters: {'solver': 'lbfgs', 'class_weight': None, 'C': 0.01}

Tuning Tuned Random Forest...
Best CV F1: 0.7855
Best parameters: {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 8}

Tuning Tuned Gradient Boosting...
Best CV F1: 0.7836
Best parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 2, 'learning_rate': 0.03}

Tuned model results:


,Model,Best_CV_F1,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Accuracy,ROC_AUC
0,Tuned Logistic Regression,0.7785,0.6006,0.3154,0.4136,0.6945,0.8813,0.7768,0.5952,0.6767,0.6894
2,Tuned Gradient Boosting,0.7836,0.6310,0.1556,0.2497,0.6649,0.9485,0.7817,0.5157,0.6619,0.6728
1,Tuned Random Forest,0.7855,0.6365,0.1326,0.2195,0.6609,0.9571,0.7819,0.5007,0.6590,0.6723


Additional Tuned Model — HistGradientBoosting




In [9]:
hist_gb = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE
)

hist_params = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_iter": [100, 150],
    "max_leaf_nodes": [7, 15, 31],
    "max_depth": [None, 3, 5],
    "min_samples_leaf": [10, 20, 30],
    "l2_regularization": [0.0, 0.5, 1.0]
}

hist_search = RandomizedSearchCV(
    estimator=hist_gb,
    param_distributions=hist_params,
    n_iter=6,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True
)

print("Tuning HistGradientBoosting...")
hist_search.fit(X_train, y_train)

best_hist_gb = hist_search.best_estimator_

print("Best CV F1:", round(hist_search.best_score_, 4))
print("Best parameters:", hist_search.best_params_)

hist_metrics = evaluate_model(
    best_hist_gb,
    X_test,
    y_test,
    threshold=0.50
)

hist_metrics["Model"] = "HistGradientBoosting"
hist_metrics["Approach"] = "Hyperparameter Tuned"
hist_metrics["Best_CV_F1"] = hist_search.best_score_

hist_df = pd.DataFrame([hist_metrics])

display(hist_df.round(4))


Tuning HistGradientBoosting...
Best CV F1: 0.7807
Best parameters: {'min_samples_leaf': 20, 'max_leaf_nodes': 15, 'max_iter': 100, 'max_depth': 5, 'learning_rate': 0.03, 'l2_regularization': 1.0}


,Accuracy,Precision,Recall,F1_Score,ROC_AUC,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Weighted_F1,Threshold,Model,Approach,Best_CV_F1
0,0.6701,0.6824,0.9039,0.7777,0.6828,0.6023,0.2571,0.3604,0.6824,0.9039,0.7777,0.5691,0.6268,0.5,HistGradientBoosting,Hyperparameter Tuned,0.7807


# Part D — Combine Experiments

All baseline, SMOTE, and tuned models are combined into one comparison table.



In [10]:
all_results = pd.concat(
    [
        baseline_df,
        smote_df,
        tuning_df,
        hist_df
    ],
    ignore_index=True
)

all_results = all_results.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

comparison_columns = [
    "Model",
    "Approach",
    "Threshold",
    "Rejected_Precision",
    "Rejected_Recall",
    "Rejected_F1",
    "Accepted_Precision",
    "Accepted_Recall",
    "Accepted_F1",
    "Macro_F1",
    "Weighted_F1",
    "Accuracy",
    "ROC_AUC"
]

display(
    all_results[comparison_columns].round(4)
)


,Model,Approach,Threshold,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Weighted_F1,Accuracy,ROC_AUC
0,SMOTE Logistic Regression,SMOTE,0.5,0.4981,0.6377,0.5593,0.7562,0.6361,0.6909,0.6251,0.6434,0.6367,0.6878
1,SMOTE Gradient Boosting,SMOTE,0.5,0.5706,0.3911,0.4641,0.7074,0.8334,0.7652,0.6147,0.6564,0.6735,0.6820
2,SMOTE SVM,SMOTE,0.5,0.5190,0.4660,0.4911,0.7142,0.7555,0.7342,0.6127,0.6463,0.6508,0.6563
3,SMOTE Random Forest,SMOTE,0.5,0.5702,0.3603,0.4416,0.7003,0.8462,0.7664,0.6040,0.6490,0.6706,0.6799
4,SMOTE Decision Tree,SMOTE,0.5,0.4960,0.4699,0.4826,0.7086,0.7297,0.7190,0.6008,0.6335,0.6358,0.6391
5,Logistic Regression,Baseline,0.5,0.6014,0.3257,0.4225,0.6969,0.8778,0.7770,0.5997,0.6488,0.6782,0.6894
6,Tuned Logistic Regression,Hyperparameter Tuned,0.5,0.6006,0.3154,0.4136,0.6945,0.8813,0.7768,0.5952,0.6455,0.6767,0.6894
7,Gradient Boosting,Baseline,0.5,0.5885,0.2896,0.3882,0.6876,0.8853,0.7740,0.5811,0.6346,0.6700,0.6859
8,Random Forest,Baseline,0.5,0.5962,0.2777,0.3789,0.6860,0.8935,0.7761,0.5775,0.6325,0.6709,0.6798
9,SVM,Baseline,0.5,0.6110,0.2596,0.3644,0.6838,0.9064,0.7795,0.5720,0.6294,0.6726,0.6634


# Part E — Threshold Testing



In [11]:
# Candidate models for threshold testing
candidate_models = {}

candidate_models.update(trained_baseline)
candidate_models.update(tuned_models)
candidate_models["HistGradientBoosting"] = best_hist_gb

threshold_values = np.arange(0.30, 0.71, 0.02)

threshold_rows = []

for name, model in candidate_models.items():

    if not hasattr(model, "predict_proba"):
        continue

    probabilities = model.predict_proba(X_test)[:, 1]

    for threshold in threshold_values:

        y_pred = (probabilities >= threshold).astype(int)

        report = classification_report(
            y_test,
            y_pred,
            output_dict=True,
            zero_division=0
        )

        threshold_rows.append({
            "Model": name,
            "Threshold": float(threshold),

            "Rejected_Precision": report["0"]["precision"],
            "Rejected_Recall": report["0"]["recall"],
            "Rejected_F1": report["0"]["f1-score"],

            "Accepted_Precision": report["1"]["precision"],
            "Accepted_Recall": report["1"]["recall"],
            "Accepted_F1": report["1"]["f1-score"],

            "Macro_F1": report["macro avg"]["f1-score"],
            "Weighted_F1": report["weighted avg"]["f1-score"],
            "Accuracy": accuracy_score(y_test, y_pred)
        })

threshold_df = pd.DataFrame(threshold_rows)

best_threshold_rows = (
    threshold_df
    .sort_values(
        ["Macro_F1", "Rejected_F1"],
        ascending=[False, False]
    )
    .groupby("Model", as_index=False)
    .first()
)

print("Best threshold for each probability-based model:")
display(best_threshold_rows.round(4))


Best threshold for each probability-based model:


,Model,Threshold,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Weighted_F1,Accuracy
0,Decision Tree,0.62,0.4671,0.5237,0.4938,0.7105,0.6617,0.6852,0.5895,0.6160,0.6118
1,Gradient Boosting,0.60,0.5323,0.5331,0.5327,0.7354,0.7347,0.7351,0.6339,0.6619,0.6619
2,HistGradientBoosting,0.60,0.5324,0.5198,0.5260,0.7317,0.7415,0.7366,0.6313,0.6605,0.6614
3,Logistic Regression,0.60,0.5268,0.5445,0.5355,0.7371,0.7231,0.7300,0.6328,0.6597,0.6585
4,Random Forest,0.60,0.5187,0.5373,0.5279,0.7326,0.7178,0.7251,0.6265,0.6538,0.6525
5,SVM,0.64,0.5193,0.5110,0.5151,0.7256,0.7322,0.7289,0.6220,0.6516,0.6522
6,Tuned Gradient Boosting,0.62,0.5078,0.5445,0.5255,0.7311,0.7011,0.7158,0.6206,0.6470,0.6445
7,Tuned Logistic Regression,0.62,0.5148,0.5931,0.5512,0.7479,0.6835,0.7143,0.6327,0.6553,0.6508
8,Tuned Random Forest,0.62,0.5087,0.5373,0.5226,0.7294,0.7061,0.7176,0.6201,0.6471,0.6451


## Threshold Comparison



In [12]:
threshold_comparison = best_threshold_rows[
    [
        "Model",
        "Threshold",
        "Rejected_Precision",
        "Rejected_Recall",
        "Rejected_F1",
        "Accepted_Precision",
        "Accepted_Recall",
        "Accepted_F1",
        "Macro_F1",
        "Accuracy"
    ]
].sort_values("Macro_F1", ascending=False)

display(threshold_comparison.round(4))


,Model,Threshold,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Accuracy
1,Gradient Boosting,0.60,0.5323,0.5331,0.5327,0.7354,0.7347,0.7351,0.6339,0.6619
3,Logistic Regression,0.60,0.5268,0.5445,0.5355,0.7371,0.7231,0.7300,0.6328,0.6585
7,Tuned Logistic Regression,0.62,0.5148,0.5931,0.5512,0.7479,0.6835,0.7143,0.6327,0.6508
2,HistGradientBoosting,0.60,0.5324,0.5198,0.5260,0.7317,0.7415,0.7366,0.6313,0.6614
4,Random Forest,0.60,0.5187,0.5373,0.5279,0.7326,0.7178,0.7251,0.6265,0.6525
5,SVM,0.64,0.5193,0.5110,0.5151,0.7256,0.7322,0.7289,0.6220,0.6522
6,Tuned Gradient Boosting,0.62,0.5078,0.5445,0.5255,0.7311,0.7011,0.7158,0.6206,0.6445
8,Tuned Random Forest,0.62,0.5087,0.5373,0.5226,0.7294,0.7061,0.7176,0.6201,0.6451
0,Decision Tree,0.62,0.4671,0.5237,0.4938,0.7105,0.6617,0.6852,0.5895,0.6118


# Part F — Final Balanced Model Selection



In [13]:
# Build final candidate table from:
# 1. Baseline results at threshold 0.50
# 2. SMOTE results at threshold 0.50
# 3. Tuned results at threshold 0.50
# 4. Best threshold configurations

# Start with all baseline/SMOTE/tuned results
selection_rows = all_results.copy()

# Add threshold-optimized configurations
threshold_selection = best_threshold_rows.copy()
threshold_selection["Approach"] = "Threshold Optimized"
threshold_selection["Weighted_F1"] = threshold_selection["Weighted_F1"]
threshold_selection["ROC_AUC"] = np.nan

selection_threshold = threshold_selection[
    [
        "Model",
        "Approach",
        "Threshold",
        "Rejected_Precision",
        "Rejected_Recall",
        "Rejected_F1",
        "Accepted_Precision",
        "Accepted_Recall",
        "Accepted_F1",
        "Macro_F1",
        "Weighted_F1",
        "Accuracy",
        "ROC_AUC"
    ]
]

selection_table = pd.concat(
    [
        selection_rows[comparison_columns],
        selection_threshold
    ],
    ignore_index=True
)

selection_table = selection_table.sort_values(
    ["Macro_F1", "Rejected_F1"],
    ascending=[False, False]
).reset_index(drop=True)

print("FINAL BALANCED MODEL CANDIDATES")
display(selection_table.round(4))

best_final = selection_table.iloc[0]

print("\n" + "=" * 80)
print("SELECTED FINAL MODEL")
print("=" * 80)

print("Model:", best_final["Model"])
print("Approach:", best_final["Approach"])
print("Threshold:", best_final["Threshold"])
print("Rejected F1:", round(best_final["Rejected_F1"], 4))
print("Accepted F1:", round(best_final["Accepted_F1"], 4))
print("Macro F1:", round(best_final["Macro_F1"], 4))
print("Accuracy:", round(best_final["Accuracy"], 4))


FINAL BALANCED MODEL CANDIDATES


,Model,Approach,Threshold,Rejected_Precision,Rejected_Recall,Rejected_F1,Accepted_Precision,Accepted_Recall,Accepted_F1,Macro_F1,Weighted_F1,Accuracy,ROC_AUC
0,Gradient Boosting,Threshold Optimized,0.60,0.5323,0.5331,0.5327,0.7354,0.7347,0.7351,0.6339,0.6619,0.6619,NaN
1,Logistic Regression,Threshold Optimized,0.60,0.5268,0.5445,0.5355,0.7371,0.7231,0.7300,0.6328,0.6597,0.6585,NaN
2,Tuned Logistic Regression,Threshold Optimized,0.62,0.5148,0.5931,0.5512,0.7479,0.6835,0.7143,0.6327,0.6553,0.6508,NaN
3,HistGradientBoosting,Threshold Optimized,0.60,0.5324,0.5198,0.5260,0.7317,0.7415,0.7366,0.6313,0.6605,0.6614,NaN
4,Random Forest,Threshold Optimized,0.60,0.5187,0.5373,0.5279,0.7326,0.7178,0.7251,0.6265,0.6538,0.6525,NaN
5,SMOTE Logistic Regression,SMOTE,0.50,0.4981,0.6377,0.5593,0.7562,0.6361,0.6909,0.6251,0.6434,0.6367,0.6878
6,SVM,Threshold Optimized,0.64,0.5193,0.5110,0.5151,0.7256,0.7322,0.7289,0.6220,0.6516,0.6522,NaN
7,Tuned Gradient Boosting,Threshold Optimized,0.62,0.5078,0.5445,0.5255,0.7311,0.7011,0.7158,0.6206,0.6470,0.6445,NaN
8,Tuned Random Forest,Threshold Optimized,0.62,0.5087,0.5373,0.5226,0.7294,0.7061,0.7176,0.6201,0.6471,0.6451,NaN
9,SMOTE Gradient Boosting,SMOTE,0.50,0.5706,0.3911,0.4641,0.7074,0.8334,0.7652,0.6147,0.6564,0.6735,0.6820



SELECTED FINAL MODEL
Model: Gradient Boosting
Approach: Threshold Optimized
Threshold: 0.6000000000000003
Rejected F1: 0.5327
Accepted F1: 0.7351
Macro F1: 0.6339
Accuracy: 0.6619


## 17. Retrieve the Actual Final Model


In [14]:
# Map all trained model objects
model_registry = {}

model_registry.update(trained_baseline)
model_registry.update(tuned_models)
model_registry["HistGradientBoosting"] = best_hist_gb

# SMOTE models are also available, but their model names are unique.
model_registry.update(trained_smote)

selected_model_name = best_final["Model"]
selected_approach = best_final["Approach"]
selected_threshold = float(best_final["Threshold"])

# For SMOTE models, the names are already in the registry.
if selected_model_name not in model_registry:
    raise KeyError(
        f"Selected model '{selected_model_name}' was not found in the trained model registry."
    )

final_model = model_registry[selected_model_name]

print("Final model object:", type(final_model).__name__)
print("Selected model:", selected_model_name)
print("Approach:", selected_approach)
print("Threshold:", selected_threshold)


Final model object: GradientBoostingClassifier
Selected model: Gradient Boosting
Approach: Threshold Optimized
Threshold: 0.6000000000000003


# Part G — Save Final Model



In [15]:
final_model_path = os.path.join(
    MODELS_DIR,
    "final_model.joblib"
)

final_metadata_path = os.path.join(
    MODELS_DIR,
    "final_model_metadata.joblib"
)

final_metadata = {
    "model_name": selected_model_name,
    "approach": selected_approach,
    "threshold": selected_threshold,
    "selection_metric": "Macro F1",

    "rejected_precision": float(best_final["Rejected_Precision"]),
    "rejected_recall": float(best_final["Rejected_Recall"]),
    "rejected_f1": float(best_final["Rejected_F1"]),

    "accepted_precision": float(best_final["Accepted_Precision"]),
    "accepted_recall": float(best_final["Accepted_Recall"]),
    "accepted_f1": float(best_final["Accepted_F1"]),

    "macro_f1": float(best_final["Macro_F1"]),
    "weighted_f1": float(best_final["Weighted_F1"]),
    "accuracy": float(best_final["Accuracy"]),

    "roc_auc": (
        None
        if pd.isna(best_final["ROC_AUC"])
        else float(best_final["ROC_AUC"])
    ),

    "random_state": RANDOM_STATE
}

joblib.dump(final_model, final_model_path)
joblib.dump(final_metadata, final_metadata_path)

print("Final model saved:", final_model_path)
print("Metadata saved:", final_metadata_path)


Final model saved: d:\job acceptance prediction system\models\final_model.joblib
Metadata saved: d:\job acceptance prediction system\models\final_model_metadata.joblib


# Part H — Reload and Verify



In [16]:
reloaded_model = joblib.load(final_model_path)
reloaded_metadata = joblib.load(final_metadata_path)

if hasattr(reloaded_model, "predict_proba"):
    reload_scores = reloaded_model.predict_proba(X_test)[:, 1]
    reload_predictions = (
        reload_scores >= reloaded_metadata["threshold"]
    ).astype(int)
else:
    reload_predictions = reloaded_model.predict(X_test)

reload_report = classification_report(
    y_test,
    reload_predictions,
    output_dict=True,
    zero_division=0
)

reload_accuracy = accuracy_score(
    y_test,
    reload_predictions
)

reload_macro_f1 = reload_report["macro avg"]["f1-score"]
reload_rejected_f1 = reload_report["0"]["f1-score"]
reload_accepted_f1 = reload_report["1"]["f1-score"]

print("=" * 80)
print("FINAL MODEL RELOAD VERIFICATION")
print("=" * 80)

print("Model:", reloaded_metadata["model_name"])
print("Approach:", reloaded_metadata["approach"])
print("Threshold:", reloaded_metadata["threshold"])

print("\nAccuracy:", round(reload_accuracy, 4))
print("Rejected F1:", round(reload_rejected_f1, 4))
print("Accepted F1:", round(reload_accepted_f1, 4))
print("Macro F1:", round(reload_macro_f1, 4))

print("\nExpected Macro F1:",
      round(reloaded_metadata["macro_f1"], 4))

if abs(reload_macro_f1 - reloaded_metadata["macro_f1"]) < 1e-6:
    print("\nReload verification: PASS")
else:
    print("\nReload verification: CHECK REQUIRED")


FINAL MODEL RELOAD VERIFICATION
Model: Gradient Boosting
Approach: Threshold Optimized
Threshold: 0.6000000000000003

Accuracy: 0.6619
Rejected F1: 0.5327
Accepted F1: 0.7351
Macro F1: 0.6339

Expected Macro F1: 0.6339

Reload verification: PASS


Save Complete Comparison




In [17]:
complete_results_path = os.path.join(
    OUTPUTS_DIR,
    "complete_balanced_model_comparison.csv"
)

selection_table.to_csv(
    complete_results_path,
    index=False
)

threshold_results_path = os.path.join(
    OUTPUTS_DIR,
    "threshold_testing_results.csv"
)

threshold_df.to_csv(
    threshold_results_path,
    index=False
)

print("Saved:")
print(complete_results_path)
print(threshold_results_path)


Saved:
d:\job acceptance prediction system\outputs\complete_balanced_model_comparison.csv
d:\job acceptance prediction system\outputs\threshold_testing_results.csv
